# 19 v2 — Analyze the fine-tuned two-model diversity signal

This is a zero-simulation analysis of the two matched model experiments. The primary selector proxy
is whether lower first-chunk uncertainty selects the model that later succeeds. The two rollouts
use matched simulator identities and seeds, but run separately, so this is a signal test rather
than an online two-model policy. Whole-episode uncertainty is post-hoc only, and the oracle row
measures complementarity rather than an achievable policy.

This notebook fetches only `pi05-diversity-signal-v2-m0/m1`; v1 rows are excluded.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Fetch and validate the matched experiments

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from tqdm.auto import tqdm
from pnp.store import SupabaseStore
from pnp.diversity import (action_disagreement_summary, add_first_chunk_action_disagreement,
    analyze_diversity_signal, diversity_signal_figures, fetch_diversity_signal)

EXPERIMENT_PREFIX = "pi05-diversity-signal-v2"
OUTPUT = Path("diversity_signal_v2_outputs")
OUTPUT.mkdir(exist_ok=True)
store = SupabaseStore()
rollouts, steps = fetch_diversity_signal(
    store, experiment_prefix=EXPERIMENT_PREFIX)
expected = [f"{EXPERIMENT_PREFIX}-m0", f"{EXPERIMENT_PREFIX}-m1"]
assert sorted(rollouts.experiment.unique()) == expected
print({"rollouts": len(rollouts), "step rows": len(steps),
       "experiments": sorted(rollouts.experiment.unique())})
tables = analyze_diversity_signal(rollouts, steps)
display(tables["diversity_signal_overall"])
display(tables["diversity_signal_by_suite"])

## 3. Exact first-decision action diversity

This downloads the small generated-chunk artifacts. Disable it only for a quick table check. The
comparison uses the first 10 actions at the matched initial state. Because the simulations are
separate, this measures model plus render variability; later chunks are not compared because the
model trajectories have diverged by then.

In [ ]:
LOAD_GENERATED_CHUNKS = True
if LOAD_GENERATED_CHUNKS:
    paired = add_first_chunk_action_disagreement(
        store, tables["diversity_paired_episodes"], progress=tqdm)
    tables["diversity_paired_episodes"] = paired
    tables["diversity_action_disagreement"] = action_disagreement_summary(paired)
    display(tables["diversity_action_disagreement"])

## 4. Figures and saved tables

In [ ]:
for name, frame in tables.items(): frame.to_csv(OUTPUT / f"{name}.csv", index=False)
paths = diversity_signal_figures(tables, OUTPUT / "figures")
for path in paths:
    print(path.name); display(Image(filename=str(path)))

## 5. Concise readout

In [ ]:
row = tables["diversity_signal_overall"].iloc[0]
print("Complementarity / oracle opportunity")
print("  discordant outcomes: %.1f%% (%d/%d)" %
      (100*row.discordant_fraction, row.n_discordant, row.n_pairs))
print("  best single model SR: %.1f%%" % (100*row.best_member_sr))
print("  either-model oracle SR: %.1f%%  (gap %+0.1f pp)" %
      (100*row.oracle_either_success_sr,
       100*(row.oracle_either_success_sr-row.best_member_sr)))
print("First-observation uncertainty selector")
print("  selected SR: %.1f%%  (vs best member %+0.1f pp)" %
      (100*row.lower_first_chunk_u_sr,
       100*(row.lower_first_chunk_u_sr-row.best_member_sr)))
print("  accuracy on discordant pairs: %.1f%%; win AUC: %.3f" %
      (100*row.lower_first_chunk_u_accuracy_discordant,
       row.lower_first_chunk_u_win_auc))
print("Whole-episode selector is POST-HOC only: %.1f%%" %
      (100*row.lower_episode_u_sr_posthoc))